In [10]:
import os
import cv2
import random
import numpy as np
from ultralytics import YOLO


INPUT_DIR = "/media/ml4u/Challenge-4TB/baonhi/Places2/val_large"
OUT_IMAGES_DIR = "/media/ml4u/Challenge-4TB/baonhi/Places2/qwen_inputs_red_outline"
OUT_MASKS_DIR = "/media/ml4u/Challenge-4TB/baonhi/Places2/ground_truth_masks"
os.makedirs(OUT_IMAGES_DIR, exist_ok=True)
os.makedirs(OUT_MASKS_DIR, exist_ok=True)
model = YOLO('yolov8x-seg.pt') 

In [12]:


TARGET_CLASSES = [
    0, 1, 2, 3, 5, 7,        # Người, Xe đạp, Ô tô, Xe máy, Bus, Tải
    14, 15, 16, 17, 18, 19,  # Chim, Mèo, Chó, Ngựa, Cừu, Bò
    24, 25, 27, 28,          # Balo, Ô, Cà vạt, Vali
    39, 41,                  # Chai lọ, Cốc (Nếu đủ to sẽ được lấy)
    56, 57, 58, 59, 60, 61,  # Ghế, Sofa, Cây cảnh, Giường, Bàn ăn, Bồn cầu
    62, 63, 68, 69, 72,      # Tivi, Laptop, Lò vi sóng, Lò nướng, Tủ lạnh
    73, 74, 75               # Sách, Đồng hồ, Bình hoa
]

MIN_AREA_RATIO = 0.05
MAX_AREA_RATIO = 0.40
MAX_MASKS_PER_IMAGE = 3 # Chỉ lấy tối đa 3 mask mỗi ảnh


print(f"🚀 Bắt đầu quét thư mục: {INPUT_DIR}")
all_images = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# ĐỂ TEST: Chạy 50 ảnh đầu tiên
test_images = all_images 

total = len(test_images)
generated_pairs = 0

for count, img_name in enumerate(test_images, 1):
    img_path = os.path.join(INPUT_DIR, img_name)
    
    original_img = cv2.imread(img_path)
    if original_img is None:
        continue
        
    H, W = original_img.shape[:2]
    img_area = H * W
    
    results = model(img_path, verbose=False)
    result = results[0]
    
    if result.masks is None or result.boxes is None:
        continue
        
    valid_objects = []
    
    for i, (mask_xy, box) in enumerate(zip(result.masks.xy, result.boxes)):
        class_id = int(box.cls[0].item())
        
        if class_id not in TARGET_CLASSES:
            continue
            
        poly = np.array(mask_xy, dtype=np.int32)
        if len(poly) < 3:
            continue
            
        mask_area = cv2.contourArea(poly)
        area_ratio = mask_area / img_area
        
        if MIN_AREA_RATIO <= area_ratio <= MAX_AREA_RATIO:
            valid_objects.append({'poly': poly, 'original_index': i})
            
    if not valid_objects:
        continue 
        
    if len(valid_objects) > MAX_MASKS_PER_IMAGE:
        chosen_objects = random.sample(valid_objects, MAX_MASKS_PER_IMAGE)
    else:
        chosen_objects = valid_objects 
        
    base_name = os.path.splitext(img_name)[0]
    
    for obj in chosen_objects:
        poly = obj['poly']
        original_idx = obj['original_index']
        
        img_with_outline = original_img.copy()
        cv2.polylines(img_with_outline, [poly], isClosed=True, color=(0, 0, 255), thickness=3)
        
        binary_mask = np.zeros((H, W), dtype=np.uint8)
        cv2.fillPoly(binary_mask, [poly], 255)
        
        out_name = f"{base_name}_obj{original_idx}"
        
        cv2.imwrite(os.path.join(OUT_IMAGES_DIR, f"{out_name}.jpg"), img_with_outline)
        cv2.imwrite(os.path.join(OUT_MASKS_DIR, f"{out_name}.png"), binary_mask)
        
        generated_pairs += 1

    if count % 10 == 0:
        print(f"[{count}/{total}] Đang xử lý... Đã bốc thăm và tạo được {generated_pairs} cặp data đa dạng.")

print(f"✅ Hoàn thành chạy test! Tổng cộng tạo được {generated_pairs} cặp data.")

🚀 Bắt đầu quét thư mục: /media/ml4u/Challenge-4TB/baonhi/Places2/val_large
[30/36500] Đang xử lý... Đã bốc thăm và tạo được 19 cặp data đa dạng.
[50/36500] Đang xử lý... Đã bốc thăm và tạo được 30 cặp data đa dạng.
[90/36500] Đang xử lý... Đã bốc thăm và tạo được 61 cặp data đa dạng.
[100/36500] Đang xử lý... Đã bốc thăm và tạo được 65 cặp data đa dạng.
[120/36500] Đang xử lý... Đã bốc thăm và tạo được 78 cặp data đa dạng.
[180/36500] Đang xử lý... Đã bốc thăm và tạo được 100 cặp data đa dạng.
[190/36500] Đang xử lý... Đã bốc thăm và tạo được 107 cặp data đa dạng.
[210/36500] Đang xử lý... Đã bốc thăm và tạo được 118 cặp data đa dạng.
[250/36500] Đang xử lý... Đã bốc thăm và tạo được 135 cặp data đa dạng.
[260/36500] Đang xử lý... Đã bốc thăm và tạo được 143 cặp data đa dạng.
[290/36500] Đang xử lý... Đã bốc thăm và tạo được 160 cặp data đa dạng.
[320/36500] Đang xử lý... Đã bốc thăm và tạo được 167 cặp data đa dạng.
[360/36500] Đang xử lý... Đã bốc thăm và tạo được 186 cặp data đa dạn